# Week 39

In [1]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Only keep Arabic, Telugu and Korean examples
df_train = df_train[df_train['lang'].isin(['ar', 'te', 'ko'])]

df_train_te = df_train[df_train['lang'].isin(['te'])]

df_val_te = df_val[df_val['lang'].isin(['te'])]


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/andreasmelbye/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/andreasmelbye/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:5

## Question + Context

In [3]:
# Get the question and context seperated by [SEP] token
def get_question_context(row):
    question = row['question']
    context = row['context']
    return question + " [SEP] " + context

df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)

df_train_te = df_train_te[['input_text', 'answer', 'answer_inlang']]
df_val_te = df_val_te[['input_text', 'answer', 'answer_inlang']]


/var/folders/mn/mlngcrqj5p90g299xqtw1qch0000gn/T/ipykernel_41322/3479198400.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_te['input_text'] = df_train_te.apply(get_question_context, axis=1)
/var/folders/mn/mlngcrqj5p90g299xqtw1qch0000gn/T/ipykernel_41322/3479198400.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_val_te['input_text'] = df_val_te.apply(get_question_context, axis=1)


In [4]:
from transformers import DataCollatorForSeq2Seq
from datasets import Dataset

# Make sure the column exists
df_train_te = df_train_te[df_train_te['answer_inlang'].notnull()]
df_train_te = df_train_te[['input_text', 'answer_inlang']].rename(columns={'answer_inlang': 'target_text'})

df_val_te = df_val_te[df_val_te['answer_inlang'].notnull()]
df_val_te = df_val_te[['input_text', 'answer_inlang']].rename(columns={'answer_inlang': 'target_text'})

# Convert to HF Dataset
train_dataset = Dataset.from_pandas(df_train_te, preserve_index=False)
val_dataset = Dataset.from_pandas(df_val_te, preserve_index=False)

max_input_length = 512   # depending on context length
max_target_length = 64   # answers are short

def tokenize_function(examples):
    # Tokenize inputs (question + context)
    model_inputs = tokenizer(
        examples["input_text"], 
        max_length=max_input_length, 
        truncation=True
    )
    
    # Tokenize targets (Telugu answers)
    labels = tokenizer(
        examples["target_text"], 
        max_length=max_target_length, 
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

train_dataset = Dataset.from_pandas(df_train_te)
val_dataset = Dataset.from_pandas(df_val_te)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)


Map: 100%|██████████| 100/100 [00:00<00:00, 611.61 examples/s]


In [ ]:
df_train_te

In [5]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

import evaluate
metric = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    # Replace -100 in labels with pad_token_id
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # sacreBLEU expects list of references per prediction
    result = metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    return {"bleu": result["score"]}


In [8]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5-te-qa",
    evaluation_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    predict_with_generate=True,
    logging_dir="./logs",
    logging_steps=50,
    save_safetensors=False    
)


/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# trainer.train()


/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  elif isinstance(generation_config._eos_token_tensor, torch.Tensor):













                                             

                                         
100%|██████████| 7/7 [15:44<00:00, 43.48s/it]

                                             
100%|██████████| 7/7 [07:27<00:00, 63.99s/it]

{'eval_loss': 15.63286304473877, 'eval_bleu': 0.07847072435032314, 'eval_runtime': 49.94, 'eval_samples_per_second': 2.002, 'eval_steps_per_second': 0.26, 'epoch': 1.0}
{'train_runtime': 447.9302, 'train_samples_per_second': 0.112, 'train_steps_per_second': 0.016, 'train_loss': 20.599857875279017, 'epoch': 1.0}


TrainOutput(global_step=7, training_loss=20.599857875279017, metrics={'train_runtime': 447.9302, 'train_samples_per_second': 0.112, 'train_steps_per_second': 0.016, 'total_flos': 13373649408000.0, 'train_loss': 20.599857875279017, 'epoch': 1.0})